In [ ]:
import json, numpy as np, pandas as pd, scipy.io as sio
from pathlib import Path
from scipy import signal
from tqdm.auto import tqdm
# reuse read_pdb_mat / FNAME_RE / DAMAGE_CATALOG from notebook 01 (or import from pdb_utils.py)

ROOT   = Path("/path/to/paderborn")
EXPORT = Path("./pdb_1dcnn"); EXPORT.mkdir(exist_ok=True)
CHANNELS = ["vibration_1", "phase_current_1", "phase_current_2"]
WIN, HOP, DECIM, DTYPE = 4096, 4096, 1, np.float32
FS = 64_000 // DECIM

files = pd.read_csv("./eda_out/file_inventory.csv")
n_per_file = (int(256_000/DECIM) - WIN)//HOP + 1
tot = len(files)*n_per_file
print(f"~{n_per_file} win/file, {tot:,} windows, "
      f"{tot*len(CHANNELS)*WIN*np.dtype(DTYPE).itemsize/1e9:.2f} GB")

In [ ]:
def window_view(x, win, hop):
    n = (len(x) - win)//hop + 1
    return np.lib.stride_tricks.sliding_window_view(x, win)[::hop] if n > 0 \
           else np.empty((0, win), x.dtype)

bin_path = EXPORT/"X.bin"
meta_rows, n_written, skipped = [], 0, []

with open(bin_path, "wb") as fh:
    for fid, row in enumerate(tqdm(files.itertuples(), total=len(files))):
        try:
            ch = read_pdb_mat(row.path)
        except Exception as e:
            skipped.append((row.file_id, repr(e))); continue
        if any(c not in ch for c in CHANNELS):
            skipped.append((row.file_id, "missing channel")); continue

        sigs = []
        for c in CHANNELS:
            x = ch[c].astype(np.float32)
            if DECIM > 1:
                x = signal.decimate(x, DECIM, ftype="fir", zero_phase=True).astype(np.float32)
            sigs.append(x)
        L = min(s.size for s in sigs)
        W = np.stack([window_view(s[:L], WIN, HOP) for s in sigs], axis=1)   # (n, C, WIN)
        if W.size == 0: continue
        fh.write(np.ascontiguousarray(W, dtype=DTYPE).tobytes())

        for w in range(W.shape[0]):
            meta_rows.append(dict(idx=n_written + w, file_id=row.file_id, file_num=fid,
                                  bearing=row.bearing, cond=row.cond, trial=row.trial,
                                  rpm=row.rpm, torque=row.torque, radial_force=row.radial_force,
                                  component=row.component, origin=row.origin,
                                  severity=row.severity, win=w, start=w*HOP))
        n_written += W.shape[0]

meta = pd.DataFrame(meta_rows)
LABELS4 = {"healthy":0, "OR":1, "IR":2, "IR+OR":3}
meta["y"] = meta.component.map(LABELS4)
meta.to_parquet(EXPORT/"meta.parquet", index=False)
json.dump({"bin":"X.bin","shape":[n_written,len(CHANNELS),WIN],"dtype":str(np.dtype(DTYPE)),
           "channels":CHANNELS,"fs":FS,"win":WIN,"hop":HOP,"decim":DECIM,
           "label_map":LABELS4}, open(EXPORT/"dataset_info.json","w"), indent=2)
print(n_written, "windows;", len(skipped), "files skipped"); skipped[:5]

In [ ]:
info = json.load(open(EXPORT/"dataset_info.json"))
meta = pd.read_parquet(EXPORT/"meta.parquet")
X = np.memmap(EXPORT/"X.bin", dtype=info["dtype"], mode="r", shape=tuple(info["shape"]))

splits = json.load(open("./eda_out/splits.json"))
S = splits["artificial_to_real"]
rng = np.random.default_rng(0)
train_b = S["train"]; test_b = S["test"]
val_b   = list(rng.choice(train_b, max(2, len(train_b)//5), replace=False))
train_b = [b for b in train_b if b not in val_b]

idx = {k: meta.index[meta.bearing.isin(v)].to_numpy()
       for k, v in dict(train=train_b, val=val_b, test=test_b).items()}
np.savez(EXPORT/"split_artificial_to_real.npz", **idx)
print({k: (len(v), meta.loc[v, "y"].value_counts().to_dict()) for k, v in idx.items()})

# global per-channel stats from TRAIN ONLY (subsample for speed)
s = rng.choice(idx["train"], min(20000, len(idx["train"])), replace=False)
chunk = np.asarray(X[np.sort(s)], dtype=np.float32)
mu, sd = chunk.mean((0,2)), chunk.std((0,2)) + 1e-8
np.savez(EXPORT/"norm_stats.npz", mean=mu, std=sd); print(mu, sd)

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

class PaderbornWindows(Dataset):
    """Memmapped windows -> (C, WIN) float32 tensor, int64 label."""
    def __init__(self, root, indices, norm="global", stats=None, train=False, jitter=0):
        self.root = Path(root); self.info = json.load(open(self.root/"dataset_info.json"))
        self.shape = tuple(self.info["shape"]); self.dtype = self.info["dtype"]
        self.y = pd.read_parquet(self.root/"meta.parquet")["y"].to_numpy()
        self.idx, self.norm, self.train, self.jitter = np.asarray(indices), norm, train, jitter
        if stats is not None:
            self.mu = stats["mean"].astype(np.float32)[:, None]
            self.sd = stats["std"].astype(np.float32)[:, None]
        self._X = None                          # opened per worker -> fork-safe

    def __len__(self): return len(self.idx)

    def __getitem__(self, i):
        if self._X is None:
            self._X = np.memmap(self.root/self.info["bin"], dtype=self.dtype,
                                mode="r", shape=self.shape)
        j = int(self.idx[i])
        x = np.array(self._X[j], dtype=np.float32)                 # (C, WIN)
        if self.norm == "global":     x = (x - self.mu) / self.sd
        elif self.norm == "per_window": x = (x - x.mean(1, keepdims=True)) / (x.std(1, keepdims=True) + 1e-8)
        if self.train and self.jitter:                             # cheap augmentation
            x = np.roll(x, np.random.randint(-self.jitter, self.jitter), axis=1)
            if np.random.rand() < .5: x += np.random.normal(0, .01, x.shape).astype(np.float32)
        return torch.from_numpy(x), torch.tensor(self.y[j], dtype=torch.long)

sp = np.load(EXPORT/"split_artificial_to_real.npz"); st = np.load(EXPORT/"norm_stats.npz")
ds = {k: PaderbornWindows(EXPORT, sp[k], stats=st, train=(k=="train"), jitter=256)
      for k in ("train","val","test")}

yt = ds["train"].y[ds["train"].idx]
w = (1.0/np.bincount(yt))[yt]
dl = {"train": DataLoader(ds["train"], batch_size=256, num_workers=6, pin_memory=True,
                          persistent_workers=True,
                          sampler=WeightedRandomSampler(w, len(w), replacement=True)),
      **{k: DataLoader(ds[k], batch_size=512, shuffle=False, num_workers=4, pin_memory=True)
         for k in ("val","test")}}
xb, yb = next(iter(dl["train"])); print(xb.shape, xb.mean().item(), xb.std().item(), yb[:8])

In [ ]:
class WDCNN(nn.Module):
    def __init__(self, in_ch=3, n_classes=4, width=32):
        super().__init__()
        def blk(i, o, k=3, s=1, p=1):
            return nn.Sequential(nn.Conv1d(i, o, k, s, p, bias=False),
                                 nn.BatchNorm1d(o), nn.GELU(), nn.MaxPool1d(2))
        self.stem = nn.Sequential(nn.Conv1d(in_ch, width, 64, 16, 24, bias=False),
                                  nn.BatchNorm1d(width), nn.GELU(), nn.MaxPool1d(2))
        self.body = nn.Sequential(blk(width, width*2), blk(width*2, width*2),
                                  blk(width*2, width*4), blk(width*4, width*4))
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Dropout(0.3), nn.Linear(width*4, n_classes))
    def forward(self, x): return self.head(self.body(self.stem(x)))

dev = "cuda" if torch.cuda.is_available() else "cpu"
model = WDCNN(in_ch=len(CHANNELS), n_classes=4).to(dev)
opt = torch.optim.AdamW(model.parameters(), 1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, 3e-3, epochs=3, steps_per_epoch=len(dl["train"]))
scaler = torch.cuda.amp.GradScaler(enabled=dev=="cuda")

def evaluate(loader):
    model.eval(); c = n = 0
    with torch.no_grad():
        for x, y in loader:
            p = model(x.to(dev, non_blocking=True)).argmax(1).cpu()
            c += (p == y).sum().item(); n += y.numel()
    return c/n

for ep in range(3):
    model.train()
    for x, y in tqdm(dl["train"], desc=f"epoch {ep}"):
        x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=dev=="cuda"):
            loss = F.cross_entropy(model(x), y, label_smoothing=0.05)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
    print(f"ep{ep}: loss {loss.item():.4f} | val {evaluate(dl['val']):.3f}")
print("test (unseen bearings):", round(evaluate(dl["test"]), 3))